In [48]:
import requests

from datetime import datetime, timedelta, timezone


BASE = "https://boards-api.greenhouse.io/v1/boards"


def board_exists(company):
    url = f"{BASE}/{company}/jobs"
    r = requests.get(url)

    if r.status_code == 404:
        return None

    r.raise_for_status()
    print(f"{company} | {url}")
    return r.json()


def filter_last_24_hours(jobs):
    cutoff = datetime.now(timezone.utc) - timedelta(hours=24)

    filtered = []
    for job in jobs:
        updated_at = job.get("updated_at")
        # print(updated_at)
        if not updated_at:
            continue

        try:
            job_time = datetime.fromisoformat(updated_at.replace("Z", "+00:00"))
            if job_time >= cutoff:
                filtered.append(job)
        except Exception:
            continue

    return filtered

def filter_by_title(jobs):
    """
    Filters jobs based on title keywords.

    include_keywords: list of keywords that MUST appear (at least one)
    exclude_keywords: list of keywords that MUST NOT appear
    """
    include_keywords = ["Software", "Engineer", "Backend", "Senior"]
    exclude_keywords = ["Analyst", "Manager", "Director", "Staff", "Principal", "Android", "Frontend"]
    
    include_keywords = [k.lower() for k in (include_keywords or [])]
    exclude_keywords = [k.lower() for k in (exclude_keywords or [])]

    filtered = []

    for job in jobs:
        title = job.get("title", "")
        title_lower = title.lower()

        # Exclude check (priority)
        if any(ex_kw in title_lower for ex_kw in exclude_keywords):
            continue

        # Include check
        if include_keywords:
            if not any(in_kw in title_lower for in_kw in include_keywords):
                continue

        filtered.append(job)

    return filtered

def is_india_location(location_str):

    if not location_str:
        return False

    location_str = location_str.lower()

    india_keywords = [
        "india",
        "bangalore", "bengaluru",
        "hyderabad",
        "pune",
        "mumbai",
        "gurgaon", "gurugram", "bangkok", "thailand",
        "noida",
        "chennai"
    ]

    return any(keyword in location_str for keyword in india_keywords)


def filter_india_jobs(jobs):
    filtered = []

    for job in jobs:
        found = False

        # Case 1: Direct location
        loc = job.get("location", {}).get("name")
        if is_india_location(loc):
            filtered.append(job)
            continue

        # Case 2: Offices list
        offices = job.get("offices", [])
        for office in offices:
            loc = office.get("location", {}).get("name")
            if is_india_location(loc):
                filtered.append(job)
                found = True
                break

        if found:
            continue

    return filtered


def fetch_job_detail(company, job_id):
    url = f"{BASE}/{company}/jobs/{job_id}"
    r = requests.get(url)
    r.raise_for_status()
    return r.json()


def process_company(company):
    print(f"\nChecking {company}...")

    data = board_exists(company)

    if data is None:
        print("Not valid")
        return

    jobs = data.get("jobs", [])

    jobs = filter_last_24_hours(jobs)
    jobs = filter_by_title(jobs)
    jobs = filter_india_jobs(jobs)

    if not jobs:
        print("No valid job")

    for job in jobs:

        job_id = job["id"]

        detail = fetch_job_detail(company, job_id)

        title = detail["title"]
        location = detail.get("location", {}).get("name", "Unknown")
        updated = detail.get("updated_at", "Unknown")
        description = detail.get("content", "")

        print("\n-----------------------------")
        print("Title:", title)
        print("Location:", location)
        print("Updated:", updated)
        # print("Description:\n", description)  # truncate so your terminal doesn't explode

In [64]:
process_company("Jetbrains")


Checking Jetbrains...
Jetbrains | https://boards-api.greenhouse.io/v1/boards/Jetbrains/jobs
No valid job
